<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook A05: Forecasting Baselines</h2>
</div>

Worked solutions to the 3 exercises in
[Notebook A05: Forecasting Baselines](../notebooks/A05_Forecasting_baselines.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

The series, the split, and the baseline functions from the notebook.

In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../notebooks")      # so that nb_config resolves from here
import nb_config

sns.set_theme(style="whitegrid")

temperatures = pd.read_parquet(nb_config.CDC_TEMP_PATH)
series = temperatures["Brandenburg/Berlin"].asfreq("MS")

TEST_MONTHS = 24
SEASON_LENGTH = 12

train = series.iloc[:-TEST_MONTHS]
test = series.iloc[-TEST_MONTHS:]


def forecast_index(train, horizon):
    return pd.date_range(
        start=train.index[-1] + train.index.freq, periods=horizon, freq=train.index.freq
    )


def naive_forecast(train, horizon):
    return pd.Series(train.iloc[-1], index=forecast_index(train, horizon), name="Naive")


def seasonal_naive_forecast(train, horizon, season_length=SEASON_LENGTH):
    last_cycle = train.iloc[-season_length:].to_numpy()
    values = [last_cycle[i % season_length] for i in range(horizon)]
    return pd.Series(values, index=forecast_index(train, horizon), name="Seasonal naive")


def mean_forecast(train, horizon):
    return pd.Series(train.mean(), index=forecast_index(train, horizon), name="Mean")


def moving_average_forecast(train, horizon, window):
    return pd.Series(
        train.iloc[-window:].mean(), index=forecast_index(train, horizon), name=f"SMA({window})"
    )


def exponential_moving_average_forecast(train, horizon, span):
    smoothed = train.ewm(span=span, adjust=False).mean().iloc[-1]
    return pd.Series(smoothed, index=forecast_index(train, horizon), name=f"EMA({span})")


def mean_absolute_error(actual, forecast):
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))))


print(f"Train {len(train)} months, test {len(test)} months")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Build SMA forecasts with windows of 3, 12, and 120 months and plot them together. Which window gives the highest constant, and why? What does that tell you about choosing a window on a series with a long-term trend?

In [ ]:
windows = [3, 12, 120]
forecasts = {
    window: moving_average_forecast(train, TEST_MONTHS, window) for window in windows
}

for window, forecast in forecasts.items():
    print(f"SMA({window:3d}) = {forecast.iloc[0]:6.2f} °C   "
          f"MAE {mean_absolute_error(test, forecast):5.2f}")

print(f"\nFor reference, the mean of the whole training series: {train.mean():.2f} °C")
print(f"The last training observation is {train.index[-1].strftime('%B %Y')}: "
      f"{train.iloc[-1]:.2f} °C")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))

ax.plot(train["2020":], color="steelblue", linewidth=1.2, label="Train")
ax.plot(test, color="black", linewidth=1.8, label="Actual")

for (window, forecast), colour in zip(forecasts.items(), ["crimson", "seagreen", "darkorange"]):
    ax.plot(forecast, color=colour, linewidth=1.4, linestyle="--", label=f"SMA({window})")

ax.axvline(test.index[0], color="gray", linestyle="--", linewidth=1.0)
ax.set_title("Moving average forecasts at three window lengths", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

**The 3-month window gives much the highest constant, at 19.12 °C.**

The reason is visible in the last line of the previous cell: the training data ends in August. A 3-month
window averages June, July and August, so it is not estimating the level of the series at all — it is
reporting the temperature of the most recent summer. Any window shorter than a full cycle inherits
whatever season it happens to land in.

The 12-month and 120-month windows land within 0.06 °C of each other, at 10.40 and 10.46. Both are
multiples of twelve, so each covers whole years and the seasonal contributions cancel exactly. That is the
practical rule: **on a seasonal series, use a window that is a whole number of cycles**, or the forecast
depends on where the data happens to stop.

The trend question is the interesting half. You might expect the 120-month window, reaching back a decade,
to sit lower than the 12-month one, since the series is warming. It barely does. But compare both against
the mean of the *entire* 140-year training set, 8.86 °C, which is a degree and a half lower. The trend is
real, and it is slow enough that ten years of history is still "recent". The window length is a statement
about how much of the past you consider relevant, and on a trending series that choice moves the forecast
directly.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> Run the same comparison on a different region, for example `Bayern` or `Schleswig-Holstein`. Does the seasonal naive forecast still win? Try it also on the year-on-year differenced series (`series.diff(12).dropna()`), where the seasonality has been removed. Which baseline wins there?

In [ ]:
def score_all_baselines(train, test, season_length=SEASON_LENGTH):
    """Every baseline from the notebook, scored on the same test period."""
    horizon = len(test)
    candidates = {
        "Naive": naive_forecast(train, horizon),
        "Seasonal naive": seasonal_naive_forecast(train, horizon, season_length),
        "Mean": mean_forecast(train, horizon),
        "SMA(12)": moving_average_forecast(train, horizon, 12),
        "EMA(12)": exponential_moving_average_forecast(train, horizon, 12),
    }
    return pd.Series(
        {name: mean_absolute_error(test, forecast) for name, forecast in candidates.items()}
    ).sort_values()


comparison = {}
for region in ["Brandenburg/Berlin", "Bayern", "Schleswig-Holstein"]:
    regional = temperatures[region].asfreq("MS")
    comparison[region] = score_all_baselines(
        regional.iloc[:-TEST_MONTHS], regional.iloc[-TEST_MONTHS:]
    )

pd.DataFrame(comparison).round(2)

**Yes, the seasonal naive forecast wins in every region**, and by the same wide margin: around 1.5 °C
against 5 to 8 °C for everything else. Nothing about the original result was specific to
Brandenburg/Berlin. Every German region has the same strong, stable yearly cycle, and the same baseline
exploits it.

Now remove that cycle and run the comparison again.

In [ ]:
differenced = series.diff(SEASON_LENGTH).dropna()

differenced_scores = score_all_baselines(
    differenced.iloc[:-TEST_MONTHS], differenced.iloc[-TEST_MONTHS:]
)

print("Year-on-year differenced series:")
print(differenced_scores.round(2).to_string())

**On the differenced series the Mean wins, and the seasonal naive forecast drops to third.**

This is the point of the exercise. Differencing at lag 12 subtracts each month from the same month a year
earlier, which removes the seasonal cycle by construction. What remains is close to noise around zero, and
for a series with no structure the best constant forecast is its average — which is exactly what the Mean
baseline is.

The seasonal naive forecast, meanwhile, has lost everything it was exploiting. It now repeats last year's
*differences*, which are noise, so it scores worse than a flat line.

The general lesson is worth stating plainly: **a baseline is not good or bad in itself, it is well or badly
matched to the structure of the series**. The same five methods, ranked on the same data before and after
one transformation, come out in almost the opposite order.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-3">Exercise 3</h3>
</div>

> Take the OPS electricity consumption data (`nb_config.OPS_15M_PATH`), pick one country, and resample it to daily totals. Which of the baselines in this notebook performs best on it, and does the ranking match what you would predict from looking at the series first?

In [ ]:
ops = pd.read_parquet(nb_config.OPS_15M_PATH)


def daily_totals(country):
    """Daily consumption for one country, dropping incomplete days."""
    quarter_hourly = ops[
        (ops["country"] == country) & (ops["measure"] == "actual_entsoe_transparency")
    ]["value"].tz_convert(None)

    daily = quarter_hourly.resample("D").sum()
    return daily[daily > 0].asfreq("D")


belgium = daily_totals("BE")

print(f"{len(belgium):,} days, {belgium.index.min().date()} to {belgium.index.max().date()}")
print("\nAverage by day of week (0 = Monday):")
print((belgium.groupby(belgium.index.dayofweek).mean() / 1e6).round(2).to_string())

The weekday profile answers the "what would you predict" half before any model is fitted. Monday to
Friday sit close together, Saturday is noticeably lower and Sunday lower still. That is a **weekly** cycle,
so the seasonal naive forecast should use a season length of 7 rather than 12, and on that structure it
ought to win.

In [ ]:
WEEKLY = 7
HORIZON_DAYS = 28

belgium_train, belgium_test = belgium.iloc[:-HORIZON_DAYS], belgium.iloc[-HORIZON_DAYS:]

belgium_scores = score_all_baselines(belgium_train, belgium_test, season_length=WEEKLY)

print("Belgium, 28-day horizon (MAE as a percentage of mean daily consumption):")
print((belgium_scores / belgium_test.mean() * 100).round(1).to_string())

**The seasonal naive forecast wins, at about 1.5% of mean consumption against 6% or worse for everything
else** — and this time the season is a week, not a year. The prediction from the weekday profile was
right.

There is a trap in this exercise worth exposing, though, and it is the reason to check more than one
case.

In [ ]:
rows = []
for country in ["BE", "NL", "AT"]:
    daily = daily_totals(country)
    for horizon in [28, 90]:
        scores = score_all_baselines(
            daily.iloc[:-horizon], daily.iloc[-horizon:], season_length=WEEKLY
        )
        relative = scores / daily.iloc[-horizon:].mean() * 100
        rows.append({
            "country": country,
            "horizon": horizon,
            "best": relative.idxmin(),
            **relative.round(1).to_dict(),
        })

pd.DataFrame(rows).set_index(["country", "horizon"])

Five of these six cases put the seasonal naive forecast first, comfortably. **The Netherlands over a
28-day window does not** — there the Mean wins, and the seasonal naive comes second.

Nothing about the Netherlands is unusual. Extend the same country's test window to 90 days and the
seasonal naive forecast is back in first place by a wide margin. What changed is the amount of evidence,
not the series.

That is Notebook [A06](../notebooks/A06_Evaluating_models.ipynb)'s argument arriving early: a 28-day
window is four observations of a weekly cycle, which is not enough to rank methods whose errors differ by
a few percent. If you had picked the Netherlands, run one 28-day test, and concluded that a flat mean beats
the weekly pattern on Dutch electricity demand, you would have been confidently wrong on the strength of
one short window.

**Check more than one split before you believe a ranking**, particularly when the margin is small.

---

Back to [Notebook A05](../notebooks/A05_Forecasting_baselines.ipynb), or on to
[Notebook A06](../notebooks/A06_Evaluating_models.ipynb).